# 03 - Stage 1 pretraining (Kaggle, week 4, conditional)

**Run this only if the week-3 Stage 2 model is stable.** Plan section 13 makes Stage 1
conditional: it is the first thing cut if the timeline slips, and Stage 2 was always
designed to stand alone.

Kaggle, not Colab - 30 GPU-h/week is an explicit quota and this is the longest job.

One pretext task at this timeline: **corruption contrast**. Masked tag modelling and
geometry reconstruction are future work.

In [ ]:
# Thin by design. Everything real lives in src/mapguard/ - see CONTRIBUTING.md.
!git clone -q https://github.com/ayushmgarg/osm_mapguard.git mapguard 2>/dev/null || (cd mapguard && git pull -q)
%cd mapguard
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, "src")
print(open("results/gate1a_adiff_batch.json").read()[:200])

## Gate 1e - measure throughput before setting a budget

Edits per second on one P100 at the real batch size. The pretraining corpus size is then
derived, not guessed. 30 GPU-h/week is a ceiling, not a target.

In [ ]:
!python scripts/gate1e_throughput.py    # WS4, week 4

## Corpus C - disjointness

Corpus C must exclude every changeset id **and every uid** that appears in a Corpus D
evaluation split. Pretraining on the test set would make R4 - the most important figure
in the paper - meaningless.

In [ ]:
!python -m pytest tests/test_pretrain_eval_disjoint.py -q

## Zero-shot transfer check - before any fine-tuning

Score the frozen corruption head on the real labelled set. A model that separates
real-from-synthetic need not separate real-from-vandalism. If the AUC is near 0.5 the
corruptions are the wrong distribution and Pretext C needs redesigning.

Report the number either way. It is a finding, not a diagnostic.

In [ ]:
# from mapguard.training.stage1 import zero_shot_transfer   # WS4